In [1]:
import os
print(os.getcwd())

c:\Users\zulfi\Desktop\Human AI Interaction\Graded Project\code


In [ ]:
from pathlib import Path

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
#Scaled for 100 words for llama 3.1 8B (average model)

In [ ]:
import os
import pandas as pd
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================

os.environ["NEBIUS_API_KEY"] = "v1.CmMKHHN0YXRpY2tleS1lMDByNG13OHJlOTEwYXhtZjcSIXNlcnZpY2VhY2NvdW50LWUwMHZ2ZW53eDUwMDM1NTU1NjIMCOeCosgGEPu9g74COgsI54W6kwcQwMLuSkACWgNlMDA.AAAAAAAAAAF-s3IVuPd-6SwZfzos0vgqlAlUZtfge6Kj5JAVVepABWajqetR76LusvMMN1mo0E5Y5TbLdhzBkjxNaiMXxrQM"
MODEL_NAME = "google/gemma-2-9b-it-fast"

# CONCURRENCY SETTINGS
# Increase this if the API is fast. Decrease if you get "Rate Limit" errors.
MAX_WORKERS = 20

def build_model():
    return ChatOpenAI(
        base_url="https://api.studio.nebius.ai/v1",
        api_key=os.environ["NEBIUS_API_KEY"],
        model=MODEL_NAME,
        temperature=0.7,
        max_retries=3, # Increased retries for stability in parallel mode
        request_timeout=30
    )

# We create a thread-local model builder or just rely on LangChain's thread safety.
# For simplicity, we will instantiate the model inside the workers or pass it carefully.
# LangChain clients are generally thread-safe.
global_model = build_model()

# ==========================================
# 2. EXPERIMENT PARAMETERS
# ==========================================

# WORD_LIST = [
#     "Aberration", "Acrimony", "Adulation", "Affluence", "Alacrity", "Allegory", "Ambivalence", "Amnesty", "Anarchy", "Animosity",
#     "Anomaly", "Apathy", "Apex", "Aptitude", "Arrogance", "Atonement", "Atrophy", "Audacity", "Avarice", "Aversion",
#     "Banal", "Bane", "Benevolence", "Bias", "Bigotry", "Blasphemy", "Brevity", "Calamity", "Candor", "Catharsis",
#     "Censure", "Chaos", "Charisma", "Chivalry", "Clemency", "Coercion", "Collusion", "Complacency", "Concord", "Consensus",
#     "Contempt", "Conundrum", "Credulity", "Dearth", "Debacle", "Decorum", "Decree", "Deference", "Delusion", "Demise",
#     "Depravity", "Derision", "Despair", "Destiny", "Detriment", "Devotion", "Dilemma", "Discord", "Disdain", "Dissent",
#     "Dogma", "Drudgery", "Duplicity", "Ebullience", "Ecstasy", "Edict", "Efficacy", "Ego", "Elation", "Elegance",
#     "Empathy", "Enigma", "Enmity", "Ennui", "Epiphany", "Equity", "Essence", "Euphoria", "Exodus", "Expediency",
#     "Fallacy", "Fame", "Famine", "Fatigue", "Feud", "Fidelity", "Finesse", "Flattery", "Folly", "Fortitude",
#     "Frenzy", "Friction", "Frugality", "Futility", "Gallantry", "Gambit", "Genesis", "Glamour", "Gluttony", "Gratitude"
# ]

WORD_LIST = ["Aberration", "Acrimony", "Adulation"]

NUM_STEPS = 3
NUM_INSTANCES = 3





In [4]:
# ==========================================
# 3. CORE LOGIC (PARALLELIZED)
# ==========================================

def _worker_initial_gen(word):
    """Worker function to generate ONE initial description."""
    try:
        response = global_model.invoke([
            HumanMessage(f"Describe the object '{word}' in exactly 2 sentences without naming it directly.")
        ])
        return response.content.strip()
    except Exception as e:
        return "" # Return empty on fail, will be filtered later

def generate_initial_descriptions_parallel(word, count):
    """Generates initial descriptions using ThreadPool."""
    print(f"  > Generating {count} initial descriptions in parallel...")
    descriptions = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit 'count' tasks
        futures = [executor.submit(_worker_initial_gen, word) for _ in range(count)]

        for future in as_completed(futures):
            res = future.result()
            if res:
                descriptions.append(res)
            else:
                descriptions.append("ERROR")

    return descriptions

def _worker_process_step(desc):
    """
    Worker function to process ONE instance (Guess + Paraphrase).
    """
    if not desc or desc == "ERROR":
        return "ERROR", "ERROR"

    try:
        # 1. Probe (Guess)
        guess_resp = global_model.invoke([
            HumanMessage(f"Read this description: \"{desc}\"\nGuess the single noun being described. Reply with ONLY the word, no punctuation.")
        ])
        guess = guess_resp.content.strip().strip(".\"").lower()

        # 2. Chain (Paraphrase)
        paraphrase_resp = global_model.invoke([
            HumanMessage(f"Paraphrase the description: \"{desc}\"\nDo not explain.\nDo not guess the word.\nDo not add reasoning.\nOutput only the paraphrase.")
        ])
        new_desc = paraphrase_resp.content.strip()

        return guess, new_desc
    except Exception:
        return "ERROR", desc # Return ERROR for guess, but keep old desc to avoid breaking chain completely

def process_step_parallel(current_descriptions):
    """Runs the step for all instances in parallel."""
    guesses = [None] * len(current_descriptions)
    next_descriptions = [None] * len(current_descriptions)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Map futures to their index so we keep order (Instance 1 stays Instance 1)
        future_to_index = {
            executor.submit(_worker_process_step, desc): i
            for i, desc in enumerate(current_descriptions)
        }

        for future in as_completed(future_to_index):
            i = future_to_index[future]
            try:
                g, d = future.result()
                guesses[i] = g
                next_descriptions[i] = d
            except Exception:
                guesses[i] = "ERROR"
                next_descriptions[i] = "ERROR"

    return guesses, next_descriptions

# ==========================================
# 4. MAIN EXECUTION LOOP
# ==========================================

In [3]:
# --- SANITY CHECK ---
print("Performing API Sanity Check...")
try:
    test = global_model.invoke([HumanMessage("Say 'Hello' if you can hear me.")])
    print(f"API Check Passed. Model said: {test.content}")
except Exception as e:
    print(f"\n!!! STOPPING !!!\nThe API Key or Model is invalid. Details: {e}")
    exit()



Performing API Sanity Check...
API Check Passed. Model said: Hello! How can I assist you today? Do you have any questions or need help with anything? I'm here to help. If you have any questions about Qwen or Alibaba Cloud, feel free to ask. What would you like to know? I'm ready to assist you. Let's get started! I'm here for you. You can ask me anything, anytime. I'll do my best to provide helpful information. Let's begin! How can I help you today? I'm here to assist you with anything you need. What would you like to know or discuss? I'm always ready to help. If you have any questions or need assistance, feel free to ask.


In [6]:
all_data_records = []

print(f"\nStarting Parallel Experiment")
print(f"Words: {len(WORD_LIST)} | Instances: {NUM_INSTANCES} | Steps: {NUM_STEPS}")
print(f"Concurrency: {MAX_WORKERS} threads")

# Use TQDM to track progress across words
for word in tqdm(WORD_LIST, desc="Processing Words", unit="word"):

    # --- Step 0: Parallel Generation ---
    try:
        current_descs = generate_initial_descriptions_parallel(word, NUM_INSTANCES)
    except Exception as e:
        print(f"Skipping word '{word}' due to critical error: {e}")
        continue

    # Save Step 0
    for i, desc in enumerate(current_descs):
        all_data_records.append({
            "Word": word,
            "Instance_ID": i + 1,
            "Step": 0,
            "Description": desc,
            "Guess": word
        })

    # --- Steps 1 to 10: Parallel Loop ---
    for step_num in range(1, NUM_STEPS + 1):
        # Run parallel processing for this step
        guesses, next_descs = process_step_parallel(current_descs)

        # Save data
        for i in range(NUM_INSTANCES):
            all_data_records.append({
                "Word": word,
                "Instance_ID": i + 1,
                "Step": step_num,
                "Description": next_descs[i],
                "Guess": guesses[i]
            })

        # Update for next iteration
        current_descs = next_descs

    # Intermediate Save (Optional: saves after every word in case of crash)
    # remove this if you only want one file at the end
    if len(all_data_records) % 5000 == 0:
            pd.DataFrame(all_data_records).to_csv("semantic_drift_below_hundred_words_qwen7B.csv", index=False)




Starting Parallel Experiment
Words: 3 | Instances: 3 | Steps: 3
Concurrency: 20 threads


Processing Words:   0%|          | 0/3 [00:00<?, ?word/s]

  > Generating 3 initial descriptions in parallel...


Processing Words:  33%|███▎      | 1/3 [01:10<02:20, 70.39s/word]

  > Generating 3 initial descriptions in parallel...


Processing Words:  67%|██████▋   | 2/3 [02:21<01:10, 70.93s/word]

  > Generating 3 initial descriptions in parallel...


Processing Words: 100%|██████████| 3/3 [03:22<00:00, 67.66s/word]


In [ ]:
# --- Final Save ---
if all_data_records:
    print("\nExperiment Complete. Saving Final CSV...")
    df = pd.DataFrame(all_data_records)
    output_filename = "semantic_drift_experiment_low_freq_abstract_words_qwen7B.csv"
    df.to_csv(output_filename, index=False)
    print(f"Success! Data saved to: {os.path.abspath(output_filename)}")
    print(f"Total Rows: {len(df)}")
else:
    print("No data was generated.")

In [ ]:
# code for llama (weak model- openrouter)

In [ ]:
# ==========================================
# 0. INSTALL DEPENDENCIES
# ==========================================

import os
import pandas as pd
import time
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# ==========================================
# 1. SETUP & CONFIGURATION
# ==========================================

# !!! PASTE YOUR OPENROUTER API KEY HERE !!!
os.environ["OPENROUTER_API_KEY"] = "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"

# --- MODEL SELECTION ---
# This is the correct ID for OpenRouter
MODEL_NAME = "meta-llama/llama-3.2-3b-instruct"

# --- OUTPUT FILENAME ---
# Automatically names file: description_meta-llama_llama-3.2-3b-instruct.csv
safe_name = MODEL_NAME.replace("/", "_")
OUTPUT_FILENAME = f"description_{safe_name}.csv"

# CONCURRENCY
# OpenRouter is fast, so we can go back up to 20 workers
MAX_WORKERS = 50

def build_model():
    if "sk-or-v1" not in os.environ["OPENROUTER_API_KEY"]:
         raise ValueError("Please paste your actual OpenRouter API Key.")

    return ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
        model=MODEL_NAME,
        temperature=0.7,
        max_retries=3,
        request_timeout=60,
        # OpenRouter specific headers (optional but good practice)
        default_headers={
            "HTTP-Referer": "https://colab.research.google.com",
            "X-Title": "Semantic Drift Experiment"
        }
    )

global_model = build_model()

# ==========================================
# 2. EXPERIMENT PARAMETERS
# ==========================================


WORD_LIST = [
    "Apple", "Baby", "Ball", "Banana", "Bed", "Bird", "Boat", "Book", "Bottle", "Box",
    "Boy", "Bread", "Bus", "Cake", "Camera", "Car", "Cat", "Chair", "Chicken", "Child",
    "Clock", "Cloud", "Coat", "Coffee", "Computer", "Corn", "Cow", "Cup", "Desk", "Doctor",
    "Dog", "Door", "Dress", "Ear", "Egg", "Eye", "Face", "Farm", "Fire", "Fish",
    "Floor", "Flower", "Food", "Foot", "Fork", "Friend", "Fruit", "Garden", "Girl", "Glass",
    "Gold", "Grass", "Hair", "Hand", "Hat", "Head", "Heart", "Home", "Horse", "House",
    "Ice", "Key", "Knife", "Lamp", "Leaf", "Leg", "Letter", "Light", "Man", "Map",
    "Meat", "Milk", "Money", "Moon", "Morning", "Mother", "Mountain", "Mouse", "Mouth", "Music",
    "Night", "Nose", "Ocean", "Office", "Oil", "Paper", "Park", "Pen", "Phone", "Picture",
    "Pig", "Pizza", "Plane", "Plant", "Plate", "Rain", "Ring", "River", "Road", "Rock"
]

NUM_STEPS = 10
NUM_INSTANCES = 100

# ==========================================
# 3. CORE LOGIC (PARALLELIZED)
# ==========================================

def _worker_initial_gen(word):
    """Worker function to generate ONE initial description."""
    try:
        response = global_model.invoke([
            HumanMessage(f"Describe the object '{word}' in exactly 2 sentences without naming it directly.")
        ])
        return response.content.strip()
    except Exception as e:
        return ""

def generate_initial_descriptions_parallel(word, count):
    print(f"  > Generating {count} initial descriptions in parallel...")
    descriptions = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(_worker_initial_gen, word) for _ in range(count)]
        for future in as_completed(futures):
            res = future.result()
            if res:
                descriptions.append(res)
            else:
                descriptions.append("ERROR")

    return descriptions

def _worker_process_step(desc):
    if not desc or desc == "ERROR":
        return "ERROR", "ERROR"

    try:
        # 1. Probe (Guess)
        guess_resp = global_model.invoke([
            HumanMessage(f"Read this description: \"{desc}\"\nGuess the single noun being described. Reply with ONLY the word, no punctuation.")
        ])
        guess = guess_resp.content.strip().strip(".\"").lower()

        # 2. Chain (Paraphrase)
        paraphrase_resp = global_model.invoke([
            HumanMessage(f"Paraphrase the description: \"{desc}\"\nDo not explain.\nDo not guess the word.\nDo not add reasoning.\nOutput only the paraphrase.")
        ])
        new_desc = paraphrase_resp.content.strip()

        return guess, new_desc
    except Exception:
        return "ERROR", desc

def process_step_parallel(current_descriptions):
    guesses = [None] * len(current_descriptions)
    next_descriptions = [None] * len(current_descriptions)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_index = {
            executor.submit(_worker_process_step, desc): i
            for i, desc in enumerate(current_descriptions)
        }

        for future in as_completed(future_to_index):
            i = future_to_index[future]
            try:
                g, d = future.result()
                guesses[i] = g
                next_descriptions[i] = d
            except Exception:
                guesses[i] = "ERROR"
                next_descriptions[i] = "ERROR"

    return guesses, next_descriptions

# ==========================================
# 4. MAIN EXECUTION LOOP
# ==========================================

if __name__ == "__main__":
    print(f"Performing API Sanity Check ({MODEL_NAME})...")
    try:
        test = global_model.invoke([HumanMessage("Say 'Hello'.")])
        print(f"API Check Passed. Model said: {test.content}")
    except Exception as e:
        print(f"\n!!! STOPPING !!!\nInvalid API Key or Model Name. Details: {e}")
        exit()

    all_data_records = []
    print(f"\nStarting Experiment ({MODEL_NAME})")
    print(f"Output File: {OUTPUT_FILENAME}")

    for word in tqdm(WORD_LIST, desc="Processing Words", unit="word"):
        try:
            current_descs = generate_initial_descriptions_parallel(word, NUM_INSTANCES)
        except Exception as e:
            print(f"Skipping word '{word}' due to critical error: {e}")
            continue

        # Save Step 0
        for i, desc in enumerate(current_descs):
            all_data_records.append({
                "Word": word, "Instance_ID": i + 1, "Step": 0, "Description": desc, "Guess": word
            })

        # Steps 1 to 10
        for step_num in range(1, NUM_STEPS + 1):
            guesses, next_descs = process_step_parallel(current_descs)

            for i in range(NUM_INSTANCES):
                all_data_records.append({
                    "Word": word, "Instance_ID": i + 1, "Step": step_num,
                    "Description": next_descs[i], "Guess": guesses[i]
                })
            current_descs = next_descs

        # Intermediate Save
        if len(all_data_records) > 0:
             pd.DataFrame(all_data_records).to_csv(OUTPUT_FILENAME, index=False)

    print(f"\nExperiment Complete. Data saved to: {OUTPUT_FILENAME}")

Performing API Sanity Check (meta-llama/llama-3.2-3b-instruct)...
API Check Passed. Model said: Hello!

Starting Experiment (meta-llama/llama-3.2-3b-instruct)
Output File: description_meta-llama_llama-3.2-3b-instruct.csv


Processing Words:   0%|          | 0/100 [00:00<?, ?word/s]

  > Generating 100 initial descriptions in parallel...


Processing Words:   1%|          | 1/100 [01:29<2:28:09, 89.79s/word]

  > Generating 100 initial descriptions in parallel...


...

  > Generating 100 initial descriptions in parallel...


Processing Words:  99%|█████████▉| 99/100 [2:05:23<01:22, 82.61s/word]

  > Generating 100 initial descriptions in parallel...


Processing Words: 100%|██████████| 100/100 [2:06:39<00:00, 75.99s/word]


Experiment Complete. Data saved to: description_meta-llama_llama-3.2-3b-instruct.csv
